# CardioBench on Kaggle — CardiacNet (ASD / PAH)

Runs the CardioBench zero-shot and linear-probe protocols on CardiacNet with a free Kaggle GPU and
scores them with the benchmark's own `evaluation/cardiacnet.py` (1000× bootstrap, 95% CI).

**Before running**
1. *Add Input* → search **AbnormCardiacEchoVideos** (`xiaoweixumedicalai/abnormcardiacechovideos`) → Add.
2. *Settings* → **Accelerator: GPU T4 x2** (or P100) and **Internet: On** (needed for `git clone` and the Hugging Face weights).
3. *Run All*. Embeddings are cached in `/kaggle/working`, so re-running after an interruption resumes.

Per model it produces: `zeroshot` (prompt similarity), `probe` (logistic regression on mean-pooled
frame embeddings) and `probe-meanstd` (mean + std over frames), plus a label-efficiency curve
(5 %→100 % of the training labels, 5 seeds).

In [ ]:
# ---- configuration -------------------------------------------------------
REPO_URL = "https://github.com/emirrcodes/CardioBench"
BRANCH = "main"
MODELS = ["echo_clip", "biomed_clip", "siglip2"]   # aliases from src/models.py
DATA_ROOT = "/kaggle/input"                          # the script finds CardiacNet/ below this
OUT_DIR = "/kaggle/working/cardiacnet"
MAX_FRAMES = 16
MODELS_STR = " ".join(MODELS)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/working/CardioBench && git clone -q --depth 1 -b {BRANCH} {REPO_URL} /kaggle/working/CardioBench
# Kaggle already ships torch/transformers; only add what is missing (no torch pin).
!pip install -q "open_clip_torch>=2.31" nibabel
%cd /kaggle/working/CardioBench
!git log --oneline -1

In [ ]:
!python scripts/cardiacnet_benchmark.py \
    --data_root {DATA_ROOT} --out_dir {OUT_DIR} \
    --models {MODELS_STR} --max_frames {MAX_FRAMES} \
    --device cuda --precision fp16

## Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary = pd.read_csv(f"{OUT_DIR}/summary.csv")
cols = ["task", "model", "protocol", "n", "AUROC(%)", "AUROC(%)_ci_lo", "AUROC(%)_ci_hi",
        "Balanced_Accuracy(%)", "F1(%)"]
display(summary[cols].sort_values(["task", "AUROC(%)"], ascending=[True, False]).round(1))
display(Image(f"{OUT_DIR}/figures/auroc_by_protocol.png"))
display(Image(f"{OUT_DIR}/figures/label_efficiency.png"))

In [ ]:
# Label efficiency table: mean ± std test AUROC over seeds.
curves = pd.read_csv(f"{OUT_DIR}/label_efficiency.csv")
tab = (curves.groupby(["task", "model", "protocol", "fraction"])["auroc"]
       .agg(lambda s: f"{100*s.mean():.1f} ± {100*s.std(ddof=0):.1f}")
       .unstack("fraction"))
display(tab)

In [ ]:
# Bundle everything except the (large) embeddings for download from the Output tab.
!cd {OUT_DIR} && zip -qr /kaggle/working/cardiacnet_results.zip summary.csv label_efficiency.csv eval figures predictions curves
!ls -lh /kaggle/working/cardiacnet_results.zip